# Classification Model

Goal:

- Predict Clinton vs other

Considerations: 

- Binary reframing was required
- Severe class imbalance 
- Missing target (108 rows have no Candidate Supported value. Imputing would be fabricating labels. Drop only for this notebook and keep for other models).
- What this model actually is (not a deployment ready fraud detector, rather a demonstration of end-to-end classification on a hard, imbalanced, small-sample problem).
- Metric choice (macro F1 and ROC-AUC)


## Problem framing

The original goal is to predict which of three candidates (Clinton, Stein and Trump) an expert supported based on their survey responses. However, the dataset only contains 7 Trump supporters and 38 Stein supporters against 573 Clinton supporters. No classifier can learn a reliable decision boundary for Trump, and any apparent performance would not generalise.

The task is reframed as binary classification, ie Clinton vs Other, with other pooling Stein and Trump together . This isn't my preferred training as Stein and Trump supporters hold very different integrity views (EDA showed Stein supporters were more critical with high variance, Trump supporters rated integrity higher but at n=7 this is unreliable). Pooling loses this distinction and it's a genuine data limitation.

108 rows have no recorded 'Candidate Supported' value. These can't be imputed as fabricating labels would introduce noise directly into the target variable. They're dropped for this notebook only. regression and clustering retain them.

Class imbalance is the central challenge. A naive model that predicts Clinton for every expert achieves ~93% accuracy. Accuracy is therefore a misleading metric. This notebook uses 
macro F1 (equal weight to both classes regardless of size) and ROC-AUC (model's discriminative ability across all decision thresholds) as primary metrics.

## Load data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, f1_score, brier_score_loss, log_loss)
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import RFECV
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline as ImbPipeline

from xgboost import XGBClassifier

import shap
import warnings
warnings.filterwarnings('ignore')

c:\Users\charl\OneDrive\Documents\Personal Projects\Fraud_Detection_ML\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data and prepare target

In [2]:
df = pd.read_csv('data/survey.csv')

# Impute role columns (binary indicators, NaN means did not hold that role)
role_cols = ['Politician', 'Candidate', 'Activist', 'Monitor', 'Election Official', 'Voter', 'Citizen']
df[role_cols] = df[role_cols].fillna(0)

# Drop rows with no candidate label
clf_df = df.dropna(subset=['Candidate Supported']).copy()

# Binary target
clf_df['target'] = (clf_df['Candidate Supported'] != 'Clinton').astype(int)

print(f'Rows after dropping missing target: {len(clf_df)}')
print(f'\nClass distribution:')
print(clf_df['target'].value_counts().rename({0: 'Clinton', 1: 'Other'}))
print(f'\nClass balance: {clf_df["target"].mean():.1%} Other')

Rows after dropping missing target: 618

Class distribution:
target
Clinton    573
Other       45
Name: count, dtype: int64

Class balance: 7.3% Other
